In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)


In [0]:
df = spark.read.format("csv").option("header", True).load("/Volumes/external-catalog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
from pyspark.sql.functions import col

high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction").cast("int") < 3)
)

selected_columns = [
    "EmployeeNumber", "EmployeeName", "Department", "JobRole", "JobSatisfaction",
    "Age", "Gender", "MaritalStatus", "MonthlyIncome", "OverTime", "YearsAtCompany"
]
high_risk_selected_df = high_risk_df.select(*[c for c in selected_columns if c in high_risk_df.columns])

high_risk_selected_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees")

display(history_df.select("version", "timestamp", "operation"))

In [0]:
delta_df = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
from pyspark.sql import Row

dummy_data = [Row(
    EmployeeNumber="999999",
    Department="Dummy Dept",
    JobRole="Dummy Role",
    JobSatisfaction="1",
    Age="30",
    Gender="Other",
    MaritalStatus="Single",
    MonthlyIncome="0",
    OverTime="No",
    YearsAtCompany="0"
)]

dummy_df = spark.createDataFrame(dummy_data)

dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_risk_attrition_employees")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_attrition_employees")
display(history_df.select("version", "timestamp", "operation"))

In [0]:
delta_df = spark.read.format("delta").table("`external-catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
delta_df_v1 = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog`.default.high_risk_attrition_employees")
display(delta_df_v1)

In [0]:
delta_df_from_ts = spark.read.format("delta") \
    .option("timestampAsOf", "2025-09-20 07:45:57") \
    .table("`external-catalog`.default.high_risk_attrition_employees")
# change the timestamp above.
display(delta_df_from_ts)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data")

In [0]:
from pyspark.sql.functions import col, upper

# Example logical transformation: uppercase JobRole for all employees
transformed_df = df.withColumn("JobRole", upper(col("JobRole")))

output_path = "/Volumes/external-catalog/default/employee_transformed_data/"

transformed_df.write.mode("overwrite").partitionBy("Department").format("parquet").save(output_path)